# FreshRetailNet-LT Dataset Audit and Study Population Selection

This notebook audits the FreshRetailNet-LT data, verifies its analytical structure and data quality, and defines the final store-product study population used in the 2026 replenishment-optimization extension.

## 1. Data Setup and Schema

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)

ROOT = Path("..").resolve()
TRAIN_PATH = ROOT / "data" / "raw" / "train.parquet"
EVAL_PATH = ROOT / "data" / "raw" / "eval.parquet"

con = duckdb.connect()

print(f"Train file exists: {TRAIN_PATH.exists()}")
print(f"Eval file exists: {EVAL_PATH.exists()}")

Train file exists: True
Eval file exists: True


In [2]:
train_schema = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{TRAIN_PATH}')
    """
).df()

train_schema

,column_name,column_type,null,key,default,extra
0,city_id,BIGINT,YES,None,None,None
1,store_id,BIGINT,YES,None,None,None
2,management_group_id,BIGINT,YES,None,None,None
3,first_category_id,BIGINT,YES,None,None,None
4,second_category_id,BIGINT,YES,None,None,None
5,third_category_id,BIGINT,YES,None,None,None
6,product_id,BIGINT,YES,None,None,None
7,dt,VARCHAR,YES,None,None,None
8,sale_amount,DOUBLE,YES,None,None,None
9,hours_sale,DOUBLE[],YES,None,None,None


In [3]:
train_n = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{TRAIN_PATH}')"
).fetchone()[0]

eval_n = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{EVAL_PATH}')"
).fetchone()[0]

print(f"Train rows: {train_n:,}")
print(f"Eval rows:  {eval_n:,}")

Train rows: 7,869,549
Eval rows:  70,000


## 2. Dataset Structure and Integrity

In [4]:
overview = con.execute(
    f"""
    SELECT
        MIN(dt) AS min_date,
        MAX(dt) AS max_date,
        COUNT(DISTINCT city_id) AS n_cities,
        COUNT(DISTINCT store_id) AS n_stores,
        COUNT(DISTINCT product_id) AS n_products,
        COUNT(DISTINCT (store_id, product_id)) AS n_store_product_series,
        AVG(sale_amount) AS mean_sale_amount,
        MIN(sale_amount) AS min_sale_amount,
        MAX(sale_amount) AS max_sale_amount,
        AVG(discount) AS mean_discount,
        AVG(CASE WHEN stock_hour6_22_cnt > 0 THEN 1.0 ELSE 0.0 END)
            AS share_days_with_stockout
    FROM read_parquet('{TRAIN_PATH}')
    """
).df()

overview

,min_date,max_date,n_cities,n_stores,n_products,n_store_product_series,mean_sale_amount,min_sale_amount,max_sale_amount,mean_discount,share_days_with_stockout
0,2023-06-05,2025-07-13,18,1057,576,22939,1.685814,0.0,83.7,0.946229,0.481959


In [5]:
sample = con.execute(
    f"""
    SELECT *
    FROM read_parquet('{TRAIN_PATH}')
    LIMIT 10
    """
).df()

sample

,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,hours_sale,stock_hour6_22_cnt,hours_stock_status,activity_flag,discount,holiday_flag,precpt,avg_temperature,avg_humidity,avg_wind_level
0,0,0,2,25,62,31,54,2025-04-30,0.9,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.2, ...",5,"[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,1.000,0,1.4621,20.88,56.32,1.39
1,0,0,2,25,62,31,54,2025-05-01,2.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3, ...",10,"[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, ...",0,1.000,1,1.2773,21.52,57.65,1.63
2,0,0,2,25,62,31,54,2025-05-02,0.9,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3, ...",7,"[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,1.000,1,1.1190,21.88,60.03,1.77
3,0,0,2,25,62,31,54,2025-05-03,1.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.2, ...",0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,1.000,1,1.8706,22.63,61.44,2.32
4,0,0,2,25,62,31,54,2025-05-04,1.2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.2, ...",9,"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, ...",0,1.000,1,2.1437,21.99,67.06,2.25
5,0,0,2,25,62,31,54,2025-05-05,1.3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.2, ...",6,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,1.000,1,3.1502,23.13,65.62,2.04
6,0,0,2,25,62,31,54,2025-05-06,0.7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3,"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,1.000,0,3.8870,23.28,68.23,2.14
7,0,0,2,25,62,31,54,2025-05-07,1.2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.2, 0.1, ...",5,"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,1.000,0,3.5090,22.72,69.64,1.98
8,0,0,2,25,62,31,54,2025-05-08,1.5,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.2, ...",6,"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0.915,0,4.4019,23.55,65.20,2.14
9,0,0,2,25,62,31,54,2025-05-09,1.4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.3, 0.0, ...",5,"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,1.000,0,4.2321,23.06,64.57,2.16


In [6]:
duplicates = con.execute(
    f"""
    SELECT
        store_id,
        product_id,
        dt,
        COUNT(*) AS n
    FROM read_parquet('{TRAIN_PATH}')
    GROUP BY store_id, product_id, dt
    HAVING COUNT(*) > 1
    ORDER BY n DESC
    LIMIT 20
    """
).df()

duplicates

,store_id,product_id,dt,n


In [7]:
eval_overview = con.execute(
    f"""
    SELECT
        MIN(dt) AS min_date,
        MAX(dt) AS max_date,
        COUNT(DISTINCT store_id) AS n_stores,
        COUNT(DISTINCT product_id) AS n_products,
        COUNT(DISTINCT (store_id, product_id)) AS n_store_product_series
    FROM read_parquet('{EVAL_PATH}')
    """
).df()

eval_overview

,min_date,max_date,n_stores,n_products,n_store_product_series
0,2025-07-14,2025-07-20,982,287,10000


In [8]:
array_lengths = con.execute(
    f"""
    SELECT
        array_length(hours_sale) AS sales_len,
        array_length(hours_stock_status) AS stock_len,
        COUNT(*) AS n
    FROM read_parquet('{TRAIN_PATH}')
    GROUP BY 1, 2
    ORDER BY n DESC
    """
).df()

array_lengths

,sales_len,stock_len,n
0,24,24,7869549


In [9]:
missingness = con.execute(
    f"""
    SELECT
        AVG(CASE WHEN city_id IS NULL THEN 1.0 ELSE 0.0 END) AS city_missing,
        AVG(CASE WHEN store_id IS NULL THEN 1.0 ELSE 0.0 END) AS store_missing,
        AVG(CASE WHEN product_id IS NULL THEN 1.0 ELSE 0.0 END) AS product_missing,
        AVG(CASE WHEN dt IS NULL THEN 1.0 ELSE 0.0 END) AS date_missing,
        AVG(CASE WHEN sale_amount IS NULL THEN 1.0 ELSE 0.0 END) AS sales_missing,
        AVG(CASE WHEN stock_hour6_22_cnt IS NULL THEN 1.0 ELSE 0.0 END) AS stock_count_missing,
        AVG(CASE WHEN hours_sale IS NULL THEN 1.0 ELSE 0.0 END) AS hourly_sales_missing,
        AVG(CASE WHEN hours_stock_status IS NULL THEN 1.0 ELSE 0.0 END) AS stock_status_missing,
        AVG(CASE WHEN discount IS NULL THEN 1.0 ELSE 0.0 END) AS discount_missing,
        AVG(CASE WHEN holiday_flag IS NULL THEN 1.0 ELSE 0.0 END) AS holiday_missing,
        AVG(CASE WHEN activity_flag IS NULL THEN 1.0 ELSE 0.0 END) AS activity_missing
    FROM read_parquet('{TRAIN_PATH}')
    """
).df()

missingness

,city_missing,store_missing,product_missing,date_missing,sales_missing,stock_count_missing,hourly_sales_missing,stock_status_missing,discount_missing,holiday_missing,activity_missing
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Store-Product Series Characteristics

In [10]:
series_coverage = con.execute(
    f"""
    SELECT
        store_id,
        product_id,
        COUNT(*) AS n_days,
        MIN(dt) AS first_date,
        MAX(dt) AS last_date,
        AVG(sale_amount) AS mean_sales,
        AVG(CASE WHEN stock_hour6_22_cnt > 0 THEN 1.0 ELSE 0.0 END)
            AS stockout_day_share,
        AVG(discount) AS mean_discount,
        STDDEV_SAMP(discount) AS sd_discount
    FROM read_parquet('{TRAIN_PATH}')
    GROUP BY store_id, product_id
    """
).df()

series_coverage.describe()

,store_id,product_id,n_days,mean_sales,stockout_day_share,mean_discount,sd_discount
count,22939.000000,22939.000000,22939.000000,22939.000000,22939.000000,22939.000000,22939.000000
mean,454.716989,268.640743,343.064170,1.750788,0.490477,0.949814,0.067158
std,288.035134,172.460799,256.783197,2.243297,0.170657,0.058997,0.046959
min,0.000000,0.000000,42.000000,0.073354,0.052516,0.300000,0.000000
25%,210.000000,108.000000,112.000000,0.966125,0.368421,0.932421,0.029509
50%,427.000000,261.000000,236.000000,1.263077,0.466667,0.960453,0.065686
75%,655.000000,415.000000,607.000000,1.757143,0.590909,0.988593,0.094798
max,1056.000000,575.000000,770.000000,37.147039,1.000000,1.000000,0.341851


In [11]:
eval_series_overlap = con.execute(
    f"""
    WITH train_series AS (
        SELECT DISTINCT store_id, product_id
        FROM read_parquet('{TRAIN_PATH}')
    ),
    eval_series AS (
        SELECT DISTINCT store_id, product_id
        FROM read_parquet('{EVAL_PATH}')
    )
    SELECT
        COUNT(*) AS eval_series,
        SUM(CASE WHEN t.store_id IS NOT NULL THEN 1 ELSE 0 END)
            AS seen_in_train
    FROM eval_series e
    LEFT JOIN train_series t
      ON e.store_id = t.store_id
     AND e.product_id = t.product_id
    """
).df()

eval_series_overlap

,eval_series,seen_in_train
0,10000,10000.0


In [12]:
stockout_check = con.execute(
    f"""
    SELECT
        MIN(stock_hour6_22_cnt) AS min_stockout_hours,
        MAX(stock_hour6_22_cnt) AS max_stockout_hours,
        AVG(stock_hour6_22_cnt) AS mean_stockout_hours,
        AVG(
            CASE WHEN stock_hour6_22_cnt > 0
                 THEN 1.0 ELSE 0.0 END
        ) AS share_days_with_any_stockout
    FROM read_parquet('{TRAIN_PATH}')
    """
).df()

stockout_check

,min_stockout_hours,max_stockout_hours,mean_stockout_hours,share_days_with_any_stockout
0,0,16,3.546321,0.481959


The dataset-provided `stock_hour6_22_cnt` measures stockout hours during the
06:00–22:00 operating window. The mean of approximately 3.55 stockout hours
corresponds to about 22.2% of this 16-hour window. Separately, approximately
48.2% of store-product days contain at least one stockout hour.

In [13]:
additional_missingness = con.execute(
    f"""
    SELECT
        AVG(CASE WHEN management_group_id IS NULL THEN 1.0 ELSE 0.0 END)
            AS management_group_missing,
        AVG(CASE WHEN first_category_id IS NULL THEN 1.0 ELSE 0.0 END)
            AS first_category_missing,
        AVG(CASE WHEN second_category_id IS NULL THEN 1.0 ELSE 0.0 END)
            AS second_category_missing,
        AVG(CASE WHEN third_category_id IS NULL THEN 1.0 ELSE 0.0 END)
            AS third_category_missing,
        AVG(CASE WHEN precpt IS NULL THEN 1.0 ELSE 0.0 END)
            AS precipitation_missing,
        AVG(CASE WHEN avg_temperature IS NULL THEN 1.0 ELSE 0.0 END)
            AS temperature_missing,
        AVG(CASE WHEN avg_humidity IS NULL THEN 1.0 ELSE 0.0 END)
            AS humidity_missing,
        AVG(CASE WHEN avg_wind_level IS NULL THEN 1.0 ELSE 0.0 END)
            AS wind_missing
    FROM read_parquet('{TRAIN_PATH}')
    """
).df()

additional_missingness

,management_group_missing,first_category_missing,second_category_missing,third_category_missing,precipitation_missing,temperature_missing,humidity_missing,wind_missing
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
candidate_series = con.execute(
    f"""
    WITH eval_series AS (
        SELECT DISTINCT
            store_id,
            product_id
        FROM read_parquet('{EVAL_PATH}')
    ),
    series_stats AS (
        SELECT
            t.store_id,
            t.product_id,
            ANY_VALUE(t.management_group_id) AS management_group_id,
            ANY_VALUE(t.first_category_id) AS first_category_id,
            ANY_VALUE(t.second_category_id) AS second_category_id,
            ANY_VALUE(t.third_category_id) AS third_category_id,
            COUNT(*) AS n_days,
            MIN(CAST(t.dt AS DATE)) AS first_date,
            MAX(CAST(t.dt AS DATE)) AS last_date,
            AVG(t.sale_amount) AS mean_sales,
            AVG(
                CASE WHEN t.sale_amount > 0
                     THEN 1.0 ELSE 0.0 END
            ) AS nonzero_sales_share,
            AVG(
                CASE WHEN t.stock_hour6_22_cnt > 0
                     THEN 1.0 ELSE 0.0 END
            ) AS stockout_day_share,
            AVG(t.stock_hour6_22_cnt) AS mean_stockout_hours,
            AVG(t.discount) AS mean_discount,
            STDDEV_SAMP(t.discount) AS sd_discount
        FROM read_parquet('{TRAIN_PATH}') t
        INNER JOIN eval_series e
          ON t.store_id = e.store_id
         AND t.product_id = e.product_id
        GROUP BY
            t.store_id,
            t.product_id
    )
    SELECT
        *,
        DATE_DIFF('day', first_date, last_date) + 1 AS span_days,
        n_days * 1.0
            / (DATE_DIFF('day', first_date, last_date) + 1)
            AS coverage_rate
    FROM series_stats
    """
).df()

candidate_series.describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
)

,store_id,product_id,management_group_id,first_category_id,second_category_id,third_category_id,n_days,first_date,last_date,mean_sales,nonzero_sales_share,stockout_day_share,mean_stockout_hours,mean_discount,sd_discount,span_days,coverage_rate
count,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,468.904400,236.326900,4.97910,11.746400,36.034100,69.776300,470.692400,2024-01-25 14:20:32.640000,2025-07-13 00:00:00,1.434552,0.949086,0.488749,3.507046,0.950320,0.068895,535.402400,0.887258
min,0.000000,1.000000,0.00000,0.000000,1.000000,0.000000,42.000000,2023-06-05 00:00:00,2025-07-13 00:00:00,0.073354,0.070130,0.052516,0.446389,0.300000,0.000000,42.000000,0.144286
10%,100.000000,50.000000,2.00000,2.000000,14.000000,22.000000,78.000000,2023-06-05 00:00:00,2025-07-13 00:00:00,0.696557,0.866888,0.303026,1.906364,0.897816,0.012808,85.000000,0.519417
25%,212.000000,77.000000,5.00000,3.000000,22.000000,31.000000,124.000000,2023-06-05 00:00:00,2025-07-13 00:00:00,0.914908,0.940036,0.372727,2.362903,0.933161,0.033414,216.750000,0.913677
50%,451.000000,200.000000,6.00000,14.000000,23.000000,63.000000,507.000000,2023-07-29 00:00:00,2025-07-13 00:00:00,1.229507,0.982050,0.464257,3.043605,0.957811,0.067936,716.000000,1.000000
75%,691.000000,415.000000,6.00000,18.000000,62.000000,104.000000,770.000000,2024-12-09 06:00:00,2025-07-13 00:00:00,1.682725,0.996104,0.579228,4.240000,0.984495,0.096243,770.000000,1.000000
90%,903.000000,501.000000,6.00000,25.000000,65.000000,134.000000,770.000000,2025-04-20 00:00:00,2025-07-13 00:00:00,2.351698,1.000000,0.730250,5.673666,0.996416,0.126424,770.000000,1.000000
max,1056.000000,575.000000,6.00000,28.000000,68.000000,170.000000,770.000000,2025-06-02 00:00:00,2025-07-13 00:00:00,16.075987,1.000000,0.997403,15.023511,1.000000,0.333372,770.000000,1.000000
std,293.611963,170.257829,1.54198,8.595236,19.671584,42.254252,277.664913,NaN,NaN,0.898162,0.090785,0.158061,1.713393,0.049038,0.046346,279.389788,0.209784


## 4. Eligibility and Store Selection

In [15]:
eval_series_days = con.execute(
    f"""
    SELECT
        COUNT(*) AS n_series,
        MIN(n_days) AS min_eval_days,
        MAX(n_days) AS max_eval_days
    FROM (
        SELECT
            store_id,
            product_id,
            COUNT(*) AS n_days
        FROM read_parquet('{EVAL_PATH}')
        GROUP BY store_id, product_id
    )
    """
).df()

eval_series_days

,n_series,min_eval_days,max_eval_days
0,10000,7,7


In [16]:
eligible_series = candidate_series[
    (candidate_series["n_days"] >= 365) &
    (candidate_series["coverage_rate"] >= 0.90) &
    (candidate_series["nonzero_sales_share"] >= 0.90) &
    (candidate_series["stockout_day_share"] >= 0.20) &
    (candidate_series["stockout_day_share"] <= 0.80)
].copy()

print(f"Eligible series: {len(eligible_series):,}")
print(
    f"Share of eval universe: "
    f"{len(eligible_series) / len(candidate_series):.1%}"
)

eligible_series[
    [
        "n_days",
        "coverage_rate",
        "mean_sales",
        "nonzero_sales_share",
        "stockout_day_share",
        "mean_stockout_hours",
        "sd_discount",
    ]
].describe()

Eligible series: 4,917
Share of eval universe: 49.2%


,n_days,coverage_rate,mean_sales,nonzero_sales_share,stockout_day_share,mean_stockout_hours,sd_discount
count,4917.000000,4917.000000,4917.000000,4917.000000,4917.000000,4917.000000,4917.000000
mean,693.496848,0.986071,1.526083,0.981534,0.440356,2.754302,0.073432
std,114.541705,0.027756,0.999140,0.021617,0.126103,0.977381,0.038607
min,368.000000,0.900000,0.359613,0.900000,0.200000,0.699013,0.000000
25%,651.000000,1.000000,0.934714,0.974304,0.349112,2.110795,0.045073
50%,760.000000,1.000000,1.292857,0.989610,0.432468,2.567532,0.070618
75%,770.000000,1.000000,1.775629,0.997260,0.507846,3.198701,0.095399
max,770.000000,1.000000,15.701898,1.000000,0.800000,7.116183,0.333372


In [17]:
store_summary = (
    eligible_series
    .groupby("store_id")
    .agg(
        n_eligible_products=("product_id", "nunique"),
        n_first_categories=("first_category_id", "nunique"),
        n_second_categories=("second_category_id", "nunique"),
        median_history_days=("n_days", "median"),
        median_coverage=("coverage_rate", "median"),
        median_nonzero_sales=("nonzero_sales_share", "median"),
        median_stockout_share=("stockout_day_share", "median"),
        median_discount_sd=("sd_discount", "median"),
    )
    .sort_values(
        ["n_eligible_products", "median_history_days"],
        ascending=False
    )
)

store_summary.head(20)

,n_eligible_products,n_first_categories,n_second_categories,median_history_days,median_coverage,median_nonzero_sales,median_stockout_share,median_discount_sd
store_id,,,,,,,,
274,25,9,12,730.0,1.0,0.993179,0.379221,0.068842
155,22,6,8,758.5,1.0,0.992998,0.469023,0.055296
576,20,10,12,733.5,1.0,0.989500,0.348956,0.059397
166,19,8,9,770.0,1.0,0.993631,0.431169,0.083790
738,19,8,9,770.0,1.0,0.985623,0.452333,0.075053
139,18,10,11,737.0,1.0,0.990916,0.439488,0.052509
16,17,8,11,770.0,1.0,0.986737,0.441558,0.089310
81,17,11,12,770.0,1.0,0.988312,0.447284,0.082519
446,17,9,10,770.0,1.0,0.989610,0.474026,0.079908


In [18]:
STUDY_STORE = 274

store_products = (
    eligible_series[
        eligible_series["store_id"] == STUDY_STORE
    ]
    .sort_values(
        ["first_category_id", "second_category_id", "product_id"]
    )
    .reset_index(drop=True)
)

print(f"Eligible products in store {STUDY_STORE}: {len(store_products)}")

store_products[
    [
        "product_id",
        "management_group_id",
        "first_category_id",
        "second_category_id",
        "third_category_id",
        "n_days",
        "first_date",
        "last_date",
        "coverage_rate",
        "mean_sales",
        "nonzero_sales_share",
        "stockout_day_share",
        "mean_stockout_hours",
        "mean_discount",
        "sd_discount",
    ]
]

Eligible products in store 274: 25


,product_id,management_group_id,first_category_id,second_category_id,third_category_id,n_days,first_date,last_date,coverage_rate,mean_sales,nonzero_sales_share,stockout_day_share,mean_stockout_hours,mean_discount,sd_discount
0,59,6,3,23,49,733,2023-06-05,2025-07-13,0.951948,2.760805,0.993179,0.373806,2.758527,0.995910,0.019767
1,99,6,3,23,49,770,2023-06-05,2025-07-13,1.000000,4.070260,0.967532,0.379221,2.775325,0.959078,0.099223
2,102,6,3,23,163,598,2023-10-21,2025-07-13,0.946203,1.882993,0.979933,0.356187,2.359532,0.968824,0.068842
3,192,6,3,23,130,770,2023-06-05,2025-07-13,1.000000,2.217494,0.993506,0.355844,2.244156,0.953446,0.071607
4,277,6,3,23,104,704,2023-06-05,2025-07-13,0.914286,2.164621,0.998580,0.284091,1.761364,0.956318,0.068076
5,292,6,3,23,99,487,2024-03-14,2025-07-13,1.000000,0.972000,0.963039,0.386037,3.310062,0.928898,0.088296
6,554,6,3,23,0,368,2024-07-11,2025-07-13,1.000000,2.352500,1.000000,0.307065,1.907609,0.938451,0.077020
7,108,6,7,24,70,753,2023-06-05,2025-07-13,0.977922,2.384382,0.984064,0.456839,3.786189,0.927195,0.098741
8,128,6,7,24,70,755,2023-06-05,2025-07-13,0.980519,1.044343,0.968212,0.425166,3.562914,0.915444,0.106888
9,190,6,7,24,70,770,2023-06-05,2025-07-13,1.000000,2.234130,0.992208,0.300000,2.448052,0.970715,0.062769


In [19]:
category_composition = (
    store_products
    .groupby(
        ["first_category_id", "second_category_id"],
        as_index=False
    )
    .agg(
        n_products=("product_id", "nunique")
    )
    .sort_values(
        ["n_products", "first_category_id", "second_category_id"],
        ascending=[False, True, True]
    )
)

category_composition

,first_category_id,second_category_id,n_products
0,3,23,7
1,7,24,3
7,15,22,3
8,17,67,2
9,18,55,2
10,25,64,2
2,7,25,1
3,7,28,1
4,9,46,1
5,14,39,1


In [20]:
study_store_check = con.execute(
    f"""
    WITH selected_products AS (
        SELECT DISTINCT product_id
        FROM read_parquet('{TRAIN_PATH}')
        WHERE store_id = {STUDY_STORE}
          AND product_id IN (
              {",".join(map(str, store_products["product_id"].tolist()))}
          )
    )
    SELECT
        COUNT(DISTINCT e.product_id) AS n_eval_products,
        COUNT(*) AS n_eval_rows,
        MIN(e.dt) AS eval_start,
        MAX(e.dt) AS eval_end
    FROM read_parquet('{EVAL_PATH}') e
    INNER JOIN selected_products s
        ON e.product_id = s.product_id
    WHERE e.store_id = {STUDY_STORE}
    """
).df()

study_store_check

,n_eval_products,n_eval_rows,eval_start,eval_end
0,25,175,2025-07-14,2025-07-20


In [21]:
portfolio_path = ROOT / "data" / "processed" / "study_portfolio.csv"

store_products[
    [
        "store_id",
        "product_id",
        "management_group_id",
        "first_category_id",
        "second_category_id",
        "third_category_id",
        "n_days",
        "first_date",
        "last_date",
        "coverage_rate",
        "mean_sales",
        "nonzero_sales_share",
        "stockout_day_share",
        "mean_stockout_hours",
        "mean_discount",
        "sd_discount",
    ]
].to_csv(portfolio_path, index=False)

print("Study portfolio saved successfully.")

Study portfolio saved successfully.


## 5. Final Study Population

The final study population consists of Store 274 and all 25 of its
store-product series satisfying the pre-specified eligibility criteria:

- at least 365 training observations;
- at least 90% temporal coverage;
- nonzero sales on at least 90% of observed days;
- stockouts on 20%–80% of observed days; and
- inclusion in the seven-day evaluation period.

Store 274 was selected because it contained the largest number of eligible
products among candidate stores. All 25 eligible products were retained,
without further selection based on forecasting or evaluation performance.

The training period ends on July 13, 2025. The evaluation period,
July 14–20, 2025, is preserved as a seven-day temporal holdout.